# READ THE BRONZE DATA CREATE DATA FRAME 

In [2]:
customers_raw = spark.read.parquet("abfss://ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/customers.parquet")
orders_raw = spark.read.parquet("abfss://ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/orders.parquet")
payments_raw = spark.read.parquet("abfss://ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/payments.parquet")
support_raw = spark.read.parquet("abfss://ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/support_tickets.parquet")
web_raw = spark.read.parquet("abfss://ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/web_activities.parquet")

StatementMeta(, 1b108451-d9b2-430a-b0a5-7dd1356f65ad, 4, Finished, Available, Finished, False)

# read bronze data

In [3]:
display(customers_raw.limit(5))

StatementMeta(, 1b108451-d9b2-430a-b0a5-7dd1356f65ad, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b11e4055-0b09-4c16-aeb2-d2b1b4bca6f5)

In [4]:
display(orders_raw.limit(5))

StatementMeta(, 1b108451-d9b2-430a-b0a5-7dd1356f65ad, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c303bbaa-61ff-4ee8-a2e4-cb37d73fb7b5)

In [5]:
display(payments_raw.limit(5))

StatementMeta(, 1b108451-d9b2-430a-b0a5-7dd1356f65ad, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1cc88e44-6caf-49a1-8172-a1528bc23e4b)

In [6]:
display(support_raw.limit(5))

StatementMeta(, 3f5d9c95-3aa4-44b6-bb16-9eb93ebfc4cf, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 41b04417-ae0d-461d-b84b-dcc9f04af727)

In [6]:
display(web_raw.limit(5))

StatementMeta(, 1b108451-d9b2-430a-b0a5-7dd1356f65ad, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 92d20975-c55f-47c1-a93c-f3264156cb62)

# Create delta bronze table

In [7]:
# Save as Bronze Delta Tables
customers_raw.write.format("delta").mode("overwrite").saveAsTable("customers")
orders_raw.write.format("delta").mode("overwrite").saveAsTable("orders")
payments_raw.write.format("delta").mode("overwrite").saveAsTable("payments")
support_raw.write.format("delta").mode("overwrite").saveAsTable("support")
web_raw.write.format("delta").mode("overwrite").saveAsTable("web")

StatementMeta(, 1b108451-d9b2-430a-b0a5-7dd1356f65ad, 9, Finished, Available, Finished, False)

# cleaned the data - silver 

## customer data -clean

In [8]:
display(customers_raw.limit(5))

StatementMeta(, 1b108451-d9b2-430a-b0a5-7dd1356f65ad, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 83896966-8d0a-44ef-8e87-198142e5d0f5)

In [9]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


customers_clean = (
    customers_raw
    .withColumn("email", lower(trim(col("EMAIL"))))
    .withColumn("name", initcap(trim(col("name"))))
    .withColumn("gender", when(lower(col("gender")).isin("f", "female"), "Female")
                          .when(lower(col("gender")).isin("m", "male"), "Male")
                          .otherwise("Other"))
    .withColumn("dob", to_date(regexp_replace(col("dob"), "/", "-")))
    .withColumn("location", initcap(col("location")))
    .dropDuplicates(["customer_id"])
    .dropna(subset=["customer_id", "email"])
)
display(customers_clean.limit(6))

customers_clean.write.format("delta").mode("overwrite").saveAsTable("silver_customers")



StatementMeta(, 1b108451-d9b2-430a-b0a5-7dd1356f65ad, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4860cc87-536c-421b-966e-1d48a0d8efe3)

## clean order table

In [10]:
%%sql 
select * from orders limit 5

StatementMeta(, 1b108451-d9b2-430a-b0a5-7dd1356f65ad, 12, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 5 fields>

In [11]:
orders = spark.table("orders")
orders_clean = (
    orders
    .withColumn("order_date", 
                when(col("order_date").rlike("^\d{4}/\d{2}/\d{2}$"), to_date(col("order_date"), "yyyy/MM/dd"))
                .when(col("order_date").rlike("^\d{2}-\d{2}-\d{4}$"), to_date(col("order_date"), "dd-MM-yyyy"))
                .when(col("order_date").rlike("^\d{8}$"), to_date(col("order_date"), "yyyyMMdd"))
                .otherwise(to_date(col("order_date"), "yyyy-MM-dd")))
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .withColumn("status", initcap(col("status")))
    .dropna(subset=["customer_id", "order_date"])
    .dropDuplicates(["order_id"])
)
display(orders_clean.limit(5))
orders_clean.write.format("delta").mode("overwrite").saveAsTable("silver_orders")

StatementMeta(, 1b108451-d9b2-430a-b0a5-7dd1356f65ad, 13, Finished, Available, Finished, False)

<>:5: SyntaxWarning: invalid escape sequence '\d'
<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:7: SyntaxWarning: invalid escape sequence '\d'
<>:5: SyntaxWarning: invalid escape sequence '\d'
<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:7: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_6030/1619383925.py:5: SyntaxWarning: invalid escape sequence '\d'
  when(col("order_date").rlike("^\d{4}/\d{2}/\d{2}$"), to_date(col("order_date"), "yyyy/MM/dd"))
/tmp/ipykernel_6030/1619383925.py:6: SyntaxWarning: invalid escape sequence '\d'
  .when(col("order_date").rlike("^\d{2}-\d{2}-\d{4}$"), to_date(col("order_date"), "dd-MM-yyyy"))
/tmp/ipykernel_6030/1619383925.py:7: SyntaxWarning: invalid escape sequence '\d'
  .when(col("order_date").rlike("^\d{8}$"), to_date(col("order_date"), "yyyyMMdd"))


SynapseWidget(Synapse.DataFrame, 1072b668-32f3-4f0e-ae9b-811cea1a3f51)

# Payments table - clean

In [12]:
%%sql 
select * from payments limit 5

StatementMeta(, 1b108451-d9b2-430a-b0a5-7dd1356f65ad, 14, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 6 fields>

In [14]:
payments = spark.table("payments")
payments_clean = (
    payments
    .withColumn("payment_date", to_date(regexp_replace(col("payment_date"), "/", "-")))
    .withColumn("payment_method", initcap(col("payment_method")))
    .replace({"creditcard": "Credit Card"}, subset=["payment_method"])
    .withColumn("payment_status", initcap(col("payment_status")))
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .dropna(subset=["customer_id", "payment_date", "amount"])
)
display(payments_clean.limit(5))
payments_clean.write.format("delta").mode("overwrite").saveAsTable("silver_payments")



StatementMeta(, 1b108451-d9b2-430a-b0a5-7dd1356f65ad, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 060fc93e-1302-4227-8c4d-3ddb204bca9c)

# clean - support tables

In [15]:
%%sql
select * from support limit 5

StatementMeta(, 1b108451-d9b2-430a-b0a5-7dd1356f65ad, 17, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 5 fields>

In [16]:
support = spark.table("support")
support_clean = (
    support
    .withColumn("ticket_date", to_date(regexp_replace(col("ticket_date"), "/", "-")))
    .withColumn("issue_type", initcap(trim(col("issue_type"))))
    .withColumn("resolution_status", initcap(trim(col("resolution_status"))))
    .replace({"NA": None, "": None}, subset=["issue_type", "resolution_status"])
    .dropDuplicates(["ticket_id"])
    .dropna(subset=["customer_id", "ticket_date"])
)
display(support_clean.limit(5))

support_clean.write.format("delta").mode("overwrite").saveAsTable("silver_support")

StatementMeta(, 1b108451-d9b2-430a-b0a5-7dd1356f65ad, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 46ce200e-c41b-49a3-9bea-5f84f8c53534)

# - clean - web data 

In [17]:
%%sql 
select * from web limit 5

StatementMeta(, 1b108451-d9b2-430a-b0a5-7dd1356f65ad, 19, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 5 fields>

In [18]:
web = spark.table("web")
web_clean = (
    web
    .withColumn("session_time", to_date(regexp_replace(col("session_time"), "/", "-")))
    .withColumn("page_viewed", lower(col("page_viewed")))
    .withColumn("device_type", initcap(col("device_type")))
    .dropDuplicates(["session_id"])
    .dropna(subset=["customer_id", "session_time", "page_viewed"])
)
display(web_clean.limit(5))
web_clean.write.format("delta").mode("overwrite").saveAsTable("silver_web")


StatementMeta(, 1b108451-d9b2-430a-b0a5-7dd1356f65ad, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 03dedc0a-daa5-410e-8d1d-cc293eba7df2)

# GOLD TABLES - AGGRGATE TABLES

In [5]:
from pyspark.sql.functions import col

cust = spark.table("silver_customers").alias("c")
orders = spark.table("silver_orders").alias("o")
payments = spark.table("silver_payments").alias("p")
support = spark.table("silver_support").alias("s")
web = spark.table("silver_web").alias("w")

customer360 = (
    cust
    .join(orders, "customer_id", "left")
    .join(payments, "customer_id", "left")
    .join(support, "customer_id", "left")
    .join(web, "customer_id", "left")
    .select( 
        col("c.customer_id"),
        col("c.name"),
        col("c.email"),
        col("c.gender"),
        col("c.dob"),
        col("c.location"),

        col("o.order_id"),
        col("o.order_date"),
        col("o.amount").alias("order_amount"),
        col("o.status").alias("order_status"),

        col("p.payment_method"),
        col("p.payment_status"),
        col("p.amount").alias("payment_amount"),

        col("s.ticket_id"),
        col("s.issue_type"),
        col("s.ticket_date"),
        col("s.resolution_status"),

        col("w.page_viewed"),
        col("w.device_type"),
        col("w.session_time")
    )
)
display(customer360.limit(10))

customer360.write.format("delta").mode("overwrite").saveAsTable("gold_customer360")

StatementMeta(, 5d70846d-15ba-47b1-8f91-1511c9626623, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b4994941-c69e-4da9-ab43-7e03f5f51b41)